# 00 - Ubuntu and workspace

Read this notebook and paste each Bash block into an Ubuntu terminal, in order. Keep the same terminal for the commands in this notebook. These are shell instructions; do not run them in a Python kernel.

Place the complete project folder in a directory you own. The default is `~/workspace`; another directory is supported using `WORKSPACE_HOME`. Later containers use the stable project path `/workspace`.

Layout: `00-setup/`, `notebooks/`, `python/`, `pyspark/`, `sql/trino/`, `airflow/dags/` and `validations/`. Runtime files use ignored `.state/`.

## Inspect Ubuntu

```bash
cat /etc/os-release
```



## Check architecture

```bash
uname -m
```

This initial application stack is validated on x86_64 / AMD64. Stop here on another architecture.

## Check CPUs

```bash
nproc
```

Use at least four available CPUs for this configuration.

## Check memory

```bash
free -h
```

The initial profile uses 8 GiB. Leave memory for Ubuntu and other applications; the validation host has 16 GiB.

## Check disk space

```bash
df -h "$HOME"
```

Allow at least 40 GiB free for the initial deployment. A later shared-image build needs more space.

## One-time administrative preparation

Use sudo only for missing system packages, Docker service installation and the Docker group change below. If those changes are needed, authenticate at the beginning with:

```bash
sudo -v
```

If Docker and the required utilities already work, skip authentication and installation. There is no Minikube or kubectl Linux group to create. Install their binaries in `~/.local/bin`, add that directory to PATH, and create the cluster as your ordinary login account. Minikube creates the kubeconfig for that account.

## Inspect existing Docker

```bash
command -v docker || true
docker --version || true
systemctl is-active docker || true
```

If Docker is already healthy, skip the Docker installation blocks below.

## Install host utilities

```bash
if command -v curl >/dev/null && command -v python3 >/dev/null && test -f /etc/ssl/certs/ca-certificates.crt; then
  printf 'Host utilities are already installed.\n'
else
  sudo apt-get update
  sudo apt-get install -y ca-certificates curl python3
fi
```

These tools are used for checksums, API examples and JSON processing.

## Prepare Docker repository

```bash
sudo install -m 0755 -d /etc/apt/keyrings
sudo curl -fsSL https://download.docker.com/linux/ubuntu/gpg -o /etc/apt/keyrings/docker.asc
sudo chmod a+r /etc/apt/keyrings/docker.asc
```



## Configure Docker packages

```bash
sudo tee /etc/apt/sources.list.d/docker.sources >/dev/null <<EOF
Types: deb
URIs: https://download.docker.com/linux/ubuntu
Suites: $(. /etc/os-release && echo "${UBUNTU_CODENAME:-$VERSION_CODENAME}")
Components: stable
Architectures: $(dpkg --print-architecture)
Signed-By: /etc/apt/keyrings/docker.asc
EOF
```



## Install Docker

```bash
sudo apt-get update
sudo apt-get install -y docker-ce docker-ce-cli containerd.io docker-buildx-plugin docker-compose-plugin
sudo systemctl enable --now docker
```



## Grant Docker access once

```bash
if ! id -nG | tr ' ' '\n' | grep -qx docker; then
  sudo groupadd -f docker
  sudo usermod -aG docker "$(id -un)"
  printf 'Log out and reconnect before the next step.\n'
else
  printf 'Docker group access is already active.\n'
fi
```

If group membership was changed, stop here and reconnect. Do not use sudo for the Docker checks below. The Docker group grants access to the daemon; putting files in the workspace alone does not grant that access.

## Check ordinary-account Docker access

```bash
id -nG
docker info --format "{{.ServerVersion}}"
docker run --rm hello-world
```

Expect a Docker version and the hello-world success message. Stop if ordinary-account Docker access fails.

## Select your workspace directory

```bash
export WORKSPACE_HOME="${WORKSPACE_HOME:-$HOME/workspace}"
cd -- "$WORKSPACE_HOME"
export WORKSPACE_HOME="$PWD"
test -f versions.env
test -f platform/namespace.yaml
```

The default is `~/workspace`. For another location, first set `export WORKSPACE_HOME="$HOME/labs/workspace"` to your actual folder, then paste this block. The existing value is retained. Paths containing spaces are supported.

## Save your chosen path in .bashrc

```bash
python3 - <<'PY'
import os, shlex
from pathlib import Path
bashrc = Path.home() / '.bashrc'
text = bashrc.read_text() if bashrc.exists() else ''
kept = []
in_block = False
for line in text.splitlines():
    if line == '# BEGIN BIGDATA WORKSPACE':
        in_block = True
        continue
    if line == '# END BIGDATA WORKSPACE':
        in_block = False
        continue
    if not in_block and line != 'export WORKSPACE_HOME="$HOME/workspace"':
        kept.append(line)
block = ('\n# BEGIN BIGDATA WORKSPACE\n'
         'export WORKSPACE_HOME=' + shlex.quote(os.environ['WORKSPACE_HOME']) + '\n'
         'export PATH="$HOME/.local/bin:$PATH"\n'
         '# END BIGDATA WORKSPACE\n')
bashrc.write_text('\n'.join(kept).rstrip() + '\n' + block)
PY
export PATH="$HOME/.local/bin:$PATH"
```

This records your chosen directory. Repeating the step updates one managed block and retains other settings. New interactive Bash terminals load it; the current terminal already has the exported values.


## Load pinned tool versions

```bash
cd "$WORKSPACE_HOME"
source versions.env
test "$(uname -m)" = x86_64
LAB_ARCH=amd64
mkdir -p .state/downloads "$HOME/.local/bin"
```



## Download Minikube

```bash
curl -fL "https://github.com/kubernetes/minikube/releases/download/${LAB_MINIKUBE_VERSION}/minikube-linux-${LAB_ARCH}" -o .state/downloads/minikube
curl -fL "https://github.com/kubernetes/minikube/releases/download/${LAB_MINIKUBE_VERSION}/minikube-linux-${LAB_ARCH}.sha256" -o .state/downloads/minikube.sha256
```



## Verify Minikube checksum

```bash
(cd .state/downloads && printf "%s  minikube\n" "$(cat minikube.sha256)" | sha256sum --check)
```

Expect `minikube: OK`. Do not install if this fails.

## Install Minikube

```bash
install -m 0755 .state/downloads/minikube "$HOME/.local/bin/minikube"
minikube version
```



## Download kubectl

```bash
curl -fL "https://dl.k8s.io/release/${LAB_K8S_VERSION}/bin/linux/${LAB_ARCH}/kubectl" -o .state/downloads/kubectl
curl -fL "https://dl.k8s.io/release/${LAB_K8S_VERSION}/bin/linux/${LAB_ARCH}/kubectl.sha256" -o .state/downloads/kubectl.sha256
```



## Verify kubectl checksum

```bash
(cd .state/downloads && printf "%s  kubectl\n" "$(cat kubectl.sha256)" | sha256sum --check)
```

Expect `kubectl: OK`.

## Install kubectl

```bash
install -m 0755 .state/downloads/kubectl "$HOME/.local/bin/kubectl"
kubectl version --client
```



References: [Docker on Ubuntu](https://docs.docker.com/engine/install/ubuntu/), [Minikube](https://minikube.sigs.k8s.io/docs/start/), [kubectl on Linux](https://kubernetes.io/docs/tasks/tools/install-kubectl-linux/). Docker installation is conditional; an existing installation is reused.